# Analyze the multi-factor Kaggle run

Reads the artifacts from `examples/kaggle_training.ipynb` (downloaded into
`data/`) and analyzes them across factors: which factors ML actually helps,
the history window each one preferred, skill by horizon, and what each model
relies on. No fetching, local files only.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from climagrid.forecasting.models import LightGBMForecaster

warnings.filterwarnings("ignore")
CANDIDATES = [Path("data"), Path("../data"), Path(".")]
DATA_DIR = next((p for p in CANDIDATES if (p / "manifest.json").exists()), Path("data"))
print("data dir:", DATA_DIR.resolve())

manifest = json.loads((DATA_DIR / "manifest.json").read_text()) if (DATA_DIR / "manifest.json").exists() else {}

def load_csv(name):
    p = DATA_DIR / name
    if not p.exists():
        print("MISSING:", p)
        return None
    return pd.read_csv(p)

summary = load_csv("factor_summary.csv")
ablation = load_csv("ablation.csv")
print("factors:", manifest.get("factors"))

## 1. Cross-factor summary: where does ML earn its keep?

`recommendation = lightgbm` where the model beat persistence; `persistence`
where a one-line baseline is just as good. `history_years` is the window each
factor's backtest preferred.

In [ ]:
if summary is not None:
    show = summary.set_index("factor")
    wins = (summary["recommendation"] == "lightgbm").sum()
    print(f"ML recommended for {wins}/{len(summary)} factors")
show if summary is None else show

## 2. Did history length matter? (per-factor ablation)

Mean skill vs persistence at each history window, per factor. A flat line
means more history did not help (e.g. thermal aging); an upward line means a
factor benefits from more history (watch ice loading).

In [ ]:
if ablation is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    for tgt, g in ablation.groupby("target"):
        by_w = g.groupby("history_years")["skill_vs_persistence"].mean()
        ax.plot(by_w.index, by_w.values, marker="o", label=tgt.replace("feat_", ""))
    ax.set_xlabel("training history (years)")
    ax.set_ylabel("mean skill vs persistence")
    ax.set_title("History sensitivity by factor")
    ax.legend(fontsize=7)
    fig.tight_layout(); plt.show()

## 3. Skill by horizon, per factor
At each factor's chosen window. Positive and rising with horizon is the good
shape; flat-near-zero marks the persistence-dominated factors.

In [ ]:
if ablation is not None and summary is not None:
    chosen = dict(zip(summary["factor"], summary["history_years"]))
    fig, ax = plt.subplots(figsize=(8, 4))
    for tgt in summary["factor"]:
        rows = ablation[(ablation.target == tgt) & (ablation.history_years == chosen[tgt])]
        by_h = rows.groupby("horizon_day")["skill_vs_persistence"].mean()
        ax.plot(by_h.index, by_h.values, marker="o", label=tgt.replace("feat_", ""))
    ax.axhline(0.0, color="grey", ls="--", lw=0.8)
    ax.set_xlabel("horizon (days)"); ax.set_ylabel("skill vs persistence")
    ax.set_title("Skill by horizon, per factor")
    ax.legend(fontsize=7); fig.tight_layout(); plt.show()

## 4. What each model relies on (feature importance)
Top predictors (p50 models, averaged over horizons) for each factor where ML
is recommended.

In [ ]:
rows = []
for factor, info in manifest.get("per_factor", {}).items():
    if info.get("recommendation") != "lightgbm":
        continue
    mf = DATA_DIR / info["model_file"]
    if not mf.exists():
        continue
    m = LightGBMForecaster.load(mf)
    cfg = m._config
    imps = [m._models[(h, 0.5)].feature_importances_
            for h in range(1, cfg.horizon_days + 1) if (h, 0.5) in m._models]
    fi = pd.Series(np.mean(imps, axis=0), index=m._predictors)
    fi = (fi / fi.sum()).sort_values(ascending=False)
    rows.append({"factor": factor.replace("feat_", ""),
                 "top features": ", ".join(f"{k} {v:.2f}" for k, v in fi.head(4).items())})
pd.DataFrame(rows) if rows else "No ML-recommended models found in data/."

## Takeaways
- **Section 1** is the deploy decision: ship the model for the `lightgbm`
  factors, a baseline for the rest.
- **Section 2** shows whether the per-factor history choice mattered (and
  whether rare-event factors like ice loading wanted more years).
- **Sections 3-4** show where in the horizon the model helps and what drives
  each forecast.

All of it is environmental-stress forecasting for inspection lead time, not
failure prediction.